# XGBoost

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## Dataset

In [2]:
%cat '../00_data/Data.csv'|head

Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
1000025,5,1,1,1,2,1,3,1,1,2
1002945,5,4,4,5,7,10,3,2,1,2
1015425,3,1,1,1,2,2,3,1,1,2
1016277,6,8,8,1,3,4,3,7,1,2
1017023,4,1,1,3,2,1,3,1,1,2
1017122,8,10,10,8,7,10,9,7,1,4
1018099,1,1,1,1,2,10,3,1,1,2
1018561,2,1,2,1,2,1,3,1,1,2
1033078,2,1,1,1,2,1,1,1,5,2
cat: stdout: Broken pipe


In [3]:
data = pd.read_csv('../00_data/Data.csv')

data.head()

,Sample code number,Clump Thickness,Uniformity of Cell Size,Uniformity of Cell Shape,Marginal Adhesion,Single Epithelial Cell Size,Bare Nuclei,Bland Chromatin,Normal Nucleoli,Mitoses,Class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 683 entries, 0 to 682
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Sample code number           683 non-null    int64
 1   Clump Thickness              683 non-null    int64
 2   Uniformity of Cell Size      683 non-null    int64
 3   Uniformity of Cell Shape     683 non-null    int64
 4   Marginal Adhesion            683 non-null    int64
 5   Single Epithelial Cell Size  683 non-null    int64
 6   Bare Nuclei                  683 non-null    int64
 7   Bland Chromatin              683 non-null    int64
 8   Normal Nucleoli              683 non-null    int64
 9   Mitoses                      683 non-null    int64
 10  Class                        683 non-null    int64
dtypes: int64(11)
memory usage: 58.8 KB


In [5]:
X = data.drop(['Sample code number', 'Class'], axis=1)
y = data['Class'].values

## Splitting the data into the training and test set

In [6]:
from sklearn.model_selection import train_test_split

random_state = 0
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

## Applying classical ML models

In [7]:
from sklearn.metrics import confusion_matrix, make_scorer, f1_score, roc_auc_score
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# models configuration
models_params = {
    'Logistic Regression': {
        'model': Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(random_state=random_state))
        ]),
        'params': {
            'classifier__class_weight': [None, 'balanced']
        }
    },
    'KNN': {
        'model': Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', KNeighborsClassifier())
        ]),
        'params': {
            'classifier__n_neighbors': [3, 5, 7, 10],
            'classifier__metric': ['euclidean', 'manhattan']
        }
    },
    'Support Vector Machine': {
        'model': Pipeline([
            ("scaler", StandardScaler()), 
            ("classifier", SVC(probability=True, random_state=random_state))
        ]),
        'params':[
            {
                'classifier__kernel': ['linear'],
                'classifier__C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 5, 10],
                'classifier__class_weight': [None, 'balanced'],  # Test with and without class weighting
            },
            {
                'classifier__kernel': ['rbf'],
                'classifier__C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 5, 10],
                'classifier__gamma': [1, 0.1, 0.01, 0.001],
                'classifier__class_weight': [None, 'balanced'], 
            },
            {
                'classifier__kernel': ['poly'],
                'classifier__C': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 5, 10],
                'classifier__degree': [2, 3, 4],
                'classifier__class_weight': [None, 'balanced'], 
            }
        ]
    },
    'Naive Bayes': {
        'model': Pipeline([
            ("classifier", GaussianNB())
        ]),
        'params': {}
    },
    'Decision Tree': {
        'model': Pipeline([
            ("classifier", DecisionTreeClassifier(random_state=random_state))
        ]),
        'params': {
            'classifier__criterion': ['gini', 'entropy'],
            'classifier__class_weight': [None, 'balanced']
        }
    },
    'Random Forest': {
        'model': Pipeline([
            ("classifier", RandomForestClassifier(random_state=random_state))
        ]),
        'params': {
            'classifier__n_estimators': [10, 50, 100, 200],
            'classifier__criterion': ['gini', 'entropy'],
            'classifier__class_weight': [None, 'balanced']
        }
    }
}

cv = KFold(shuffle=True, n_splits=5, random_state=random_state)

f1_scorer = make_scorer(f1_score, pos_label=4)

print('--> Best models searching...')
# searching of the best params for every model on the training set
best_models = {}
for model_name, model_config in models_params.items():
    print(f"Run grid search to find the best params for {model_name}...")
    grid_search = GridSearchCV(model_config['model'],
                               model_config['params'],
                               cv = cv,
                               scoring=f1_scorer)
    grid_search.fit(X_train, y_train)
    best_models[model_name] = grid_search.best_estimator_
    print(f"The best model for {model_name}: {grid_search.best_params_}")

print('--> Best models evaluating...')
# evaluation of every model on the testing set
test_scores = {}
for model_name, best_model in best_models.items():
    # test the model
    y_pred = best_model.predict(X_test)
    # Predict probabilities for the positive class
    y_scores = best_model.predict_proba(X_test)[:, 1]

    f1 = f1_score(y_test, y_pred, pos_label=4)
    roc_auc = roc_auc_score(y_test, y_scores)
    cm = confusion_matrix(y_test, y_pred)
    
    test_scores[model_name] = {'f1': f1, 'roc-auc': roc_auc}
    
    print(f"{model_name}: F1-score = {f1:.4f}, ROC-AUC = {roc_auc:.4f}\n{cm}")
    
    

--> Best models searching...
Run grid search to find the best params for Logistic Regression...
The best model for Logistic Regression: {'classifier__class_weight': 'balanced'}
Run grid search to find the best params for KNN...
The best model for KNN: {'classifier__metric': 'euclidean', 'classifier__n_neighbors': 10}
Run grid search to find the best params for Support Vector Machine...
The best model for Support Vector Machine: {'classifier__C': 0.3, 'classifier__class_weight': None, 'classifier__kernel': 'linear'}
Run grid search to find the best params for Naive Bayes...
The best model for Naive Bayes: {}
Run grid search to find the best params for Decision Tree...
The best model for Decision Tree: {'classifier__class_weight': 'balanced', 'classifier__criterion': 'entropy'}
Run grid search to find the best params for Random Forest...
The best model for Random Forest: {'classifier__class_weight': 'balanced', 'classifier__criterion': 'gini', 'classifier__n_estimators': 100}
--> Best mo

## Applying XGBoost

In [8]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [9]:
from xgboost import XGBClassifier
# train XGBoost on training set
classifier = XGBClassifier()

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
y_scores = classifier.predict_proba(X_test)[:, 1]
print(f"F1-score: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_scores):.4f}")

F1-score: 0.9600
ROC-AUC: 0.9899


In [10]:
# Cross validation
from sklearn.model_selection import cross_val_score

print(f"CV Accuracy on training set: {cross_val_score(estimator=classifier, 
                                                      X=X_train, 
                                                      y=y_train, 
                                                      cv = 10).mean():.4f}")

CV Accuracy on training set: 0.9671


## Combine XGBoost and K-Fold Cross Validation and Grid Search

In [11]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

cv = KFold(shuffle=True, n_splits=10, random_state=random_state)

classifier = XGBClassifier()
grid_search = GridSearchCV(classifier, param_grid, scoring='f1', cv=cv, n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best F1-score on training set: {grid_search.best_score_:.4f}")

y_pred = grid_search.best_estimator_.predict(X_test)
y_scores = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
print(f"F1-score on test set: {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC on test set: {roc_auc_score(y_test, y_scores):.4f}")

Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best F1-score on training set: 0.9670
F1-score on test set: 0.9709
ROC-AUC on test set: 0.9961
